# Postgres to Apache Doris: built-in CDC

[Apache Doris](https://doris.apache.org/) is a columnar, MPP analytical database that speaks the
MySQL protocol. A cluster has two roles: the **FE** (frontend, Java) parses and plans SQL, keeps the
metadata and schedules jobs; the **BE** (backend, C++) stores the data and runs the queries.

Doris 4.1 can follow a Postgres database by itself: one `CREATE JOB ... ON STREAMING FROM POSTGRES`
creates the target tables, copies the existing rows and then streams inserts, updates and deletes
from the WAL. No Kafka, no Flink, no separately deployed Debezium. This notebook runs that against
the project's pg16 and checks what the announcement leaves out, next to
`logical_replication.ipynb`, which does the same with a Postgres subscriber:

- what the source sees (publication, slot) and what lands in Doris (the type mapping),
- how fast changes arrive, and what sets the pace,
- what happens to a pgvector `vector` column, and a way to keep the embeddings,
- schema changes, a paused job holding WAL, and cleanup.

Three containers from `docker/postgres-pgvector.yaml`: `pgvector-dev` (the source, `wal_level=logical`),
`doris-fe` (MySQL protocol on 9030, web UI on 8030) and `doris-be`. The images are large (4 GB FE,
9 GB BE). Out of the box the FE asks for an 8 GB heap and the BE for 90% of the machine, which on
Docker Desktop's VM ends in `MEM_LIMIT_EXCEEDED` on every query; the compose file caps them at 1 GB
heap and a 2 GB BE, about 2.5 GB for the pair at idle.

    just db doris            # starts pg16 and the Doris pair
    just nb doris_cdc
    just doris-sql           # a mysql shell on the FE

The "CDC client" that reads the WAL runs next to the BE (`be/lib/cdc_client/cdc-client.jar`); its log,
`be/log/cdc-client.log`, is the one place some of the problems below are reported.

In [ ]:
# Parameters (papermill -p)
N_ROWS = 1_000  # rows in the orders table before the job starts
N_DOCS = 200  # rows in the docs table (one embedding each)
DIM = 384  # embedding dimension, as the app's bge-small
N_BURST = 20_000  # rows written while the job is paused
MAX_INTERVAL = 1  # job property: idle wait between sync rounds, seconds (Doris default 10)

In [ ]:
import pathlib
import subprocess
import sys
import time

import numpy as np
import pandas as pd
import psycopg
import pymysql
from dotenv import load_dotenv
from pgvector.psycopg import register_vector
from psycopg.conninfo import conninfo_to_dict

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "db.py").exists())
sys.path.insert(0, str(ROOT / "src"))
load_dotenv(ROOT / ".env")

from settings import settings  # noqa: E402

SRC = settings.sqlalchemy_url("pg16").replace("postgresql+psycopg://", "postgresql://")
PG = conninfo_to_dict(SRC)
DORIS = settings.doris_url  # MySQLDsn: .host, .port, .username, .password
DORIS_DB = "pg_mirror"  # the Doris database the job writes into
SCHEMA = "doris_src"  # the Postgres schema it reads from
# the FE and BE fetch the JDBC driver themselves; a Maven Central URL saves mounting a jar
DRIVER_URL = "https://repo1.maven.org/maven2/org/postgresql/postgresql/42.7.4/postgresql-42.7.4.jar"

subprocess.run(["docker", "compose", "-f", "docker/postgres-pgvector.yaml", "--profile", "doris", "up", "-d"], cwd=ROOT, check=True, capture_output=True)


def doris_conn(database: str | None = DORIS_DB) -> pymysql.Connection:
    return pymysql.connect(host=DORIS.host, port=DORIS.port, user=DORIS.username, password=DORIS.password or "", database=database, autocommit=True)


for _ in range(60):
    try:
        with psycopg.connect(SRC, connect_timeout=2) as c:
            print(f"postgres {PG['host']}:{PG['port']}  {c.execute('SELECT version()').fetchone()[0][:18]}  wal_level={c.execute('SHOW wal_level').fetchone()[0]}")
        break
    except psycopg.OperationalError:
        time.sleep(1)
else:
    raise RuntimeError(f"Postgres at {SRC} does not answer")

for _ in range(120):
    try:
        with doris_conn(None) as c, c.cursor(pymysql.cursors.DictCursor) as cur:
            cur.execute("SHOW BACKENDS")
            backends = cur.fetchall()
        if any(b["Alive"] == "true" for b in backends):
            print(f"doris    {DORIS.host}:{DORIS.port}  {backends[0]['Version']}  alive backends={sum(b['Alive'] == 'true' for b in backends)}")
            break
    except pymysql.MySQLError:
        pass
    time.sleep(2)
else:
    raise RuntimeError("Doris FE/BE not ready: docker compose -f docker/postgres-pgvector.yaml --profile doris logs")


def q(sql: str, *args) -> pd.DataFrame:
    """One Postgres query as a DataFrame."""
    with psycopg.connect(SRC) as c:
        register_vector(c)
        cur = c.execute(sql, args or None)
        return pd.DataFrame(cur.fetchall(), columns=[col.name for col in cur.description])


def run(sql: str, *args, autocommit: bool = False) -> None:
    """One Postgres statement."""
    with psycopg.connect(SRC, autocommit=autocommit) as c:
        register_vector(c)
        c.execute(sql, args or None)


def dq(sql: str, database: str | None = DORIS_DB) -> pd.DataFrame:
    """One Doris query as a DataFrame."""
    with doris_conn(database) as c, c.cursor() as cur:
        cur.execute(sql)
        return pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])


def dx(sql: str, database: str | None = DORIS_DB) -> None:
    """One Doris statement."""
    with doris_conn(database) as c, c.cursor() as cur:
        cur.execute(sql)


def dcount(sql: str) -> int:
    """A Doris count, or -1 while the table or column does not exist yet."""
    try:
        return int(dq(sql).iloc[0, 0])
    except pymysql.MySQLError:
        return -1


def wait_for(pred, timeout: float = 60.0) -> float:
    """Seconds until pred() is true."""
    t0 = time.perf_counter()
    while time.perf_counter() - t0 < timeout:
        if pred():
            return time.perf_counter() - t0
        time.sleep(0.1)
    raise TimeoutError("Doris did not catch up")


def create_job(name: str, tables: str) -> None:
    """The one statement: follow SCHEMA.tables from Postgres into DORIS_DB."""
    dx(f"""
        CREATE JOB {name}
        PROPERTIES ("max_interval" = "{MAX_INTERVAL}")
        ON STREAMING
        FROM POSTGRES (
            "jdbc_url" = "jdbc:postgresql://db:5432/{PG['dbname']}",
            "driver_url" = "{DRIVER_URL}",
            "driver_class" = "org.postgresql.Driver",
            "user" = "{PG['user']}",
            "password" = "{PG['password']}",
            "database" = "{PG['dbname']}",
            "schema" = "{SCHEMA}",
            "include_tables" = "{tables}",
            "offset" = "initial"
        )
        TO DATABASE {DORIS_DB} (
            "table.create.properties.replication_num" = "1"
        )
    """)


def job(name: str) -> pd.DataFrame:
    return dq(f"SELECT Name, Status, SucceedTaskCount, FailedTaskCount, CurrentOffset, LoadStatistic, JobRuntimeMsg, ErrorMsg FROM jobs('type'='insert') WHERE Name = '{name}'").T

## 1. The source tables

A fresh schema, so nothing in the playground's own tables is touched. `orders` carries the types an
OLTP table usually has (uuid, numeric, jsonb, an array, timestamptz). `docs` holds a
`vector(384)` embedding plus two `real[]` copies of it, one as a generated column and one kept by a
trigger; section 5 shows why both are there and only one survives.

The job only syncs tables with a primary key, so both have one. A Doris database is the target;
anything left over from a previous run is dropped first, jobs before the database.

In [ ]:
for name in ("orders_job", "docs_job"):
    try:
        dx(f"DROP JOB WHERE jobName = '{name}'")
    except pymysql.MySQLError:
        pass
dx(f"DROP DATABASE IF EXISTS {DORIS_DB} FORCE", database=None)
dx(f"CREATE DATABASE {DORIS_DB}", database=None)

run(f"DROP SCHEMA IF EXISTS {SCHEMA} CASCADE", autocommit=True)
run(f"CREATE SCHEMA {SCHEMA}", autocommit=True)
run(f"""
    CREATE TABLE {SCHEMA}.orders (
        id bigserial PRIMARY KEY,
        customer uuid NOT NULL DEFAULT gen_random_uuid(),
        status text NOT NULL,
        amount numeric(12, 2) NOT NULL,
        payload jsonb NOT NULL,
        tags text[] NOT NULL,
        created_at timestamptz NOT NULL DEFAULT now()
    )
""", autocommit=True)
run(f"""
    INSERT INTO {SCHEMA}.orders (status, amount, payload, tags)
    SELECT (ARRAY['new', 'paid', 'shipped'])[1 + mod(g, 3)],
           round((random() * 500)::numeric, 2),
           jsonb_build_object('channel', (ARRAY['web', 'app', 'store'])[1 + mod(g, 3)], 'items', 1 + mod(g, 5)),
           ARRAY['t' || mod(g, 7)]
    FROM generate_series(1, %s) g
""", N_ROWS, autocommit=True)

run(f"""
    CREATE TABLE {SCHEMA}.docs (
        id bigserial PRIMARY KEY,
        title text NOT NULL,
        embedding vector({DIM}) NOT NULL,
        embedding_gen real[] GENERATED ALWAYS AS (embedding::real[]) STORED,
        embedding_arr real[]
    )
""", autocommit=True)
run(f"""
    CREATE FUNCTION {SCHEMA}.docs_embedding_arr() RETURNS trigger LANGUAGE plpgsql AS $$
    BEGIN
        NEW.embedding_arr := NEW.embedding::real[];
        RETURN NEW;
    END $$
""", autocommit=True)
run(f"""
    CREATE TRIGGER docs_embedding_arr BEFORE INSERT OR UPDATE OF embedding ON {SCHEMA}.docs
    FOR EACH ROW EXECUTE FUNCTION {SCHEMA}.docs_embedding_arr()
""", autocommit=True)

rng = np.random.default_rng(7)


def random_unit(n: int) -> np.ndarray:
    v = rng.standard_normal((n, DIM)).astype(np.float32)
    return v / np.linalg.norm(v, axis=1, keepdims=True)


def insert_docs(vectors: np.ndarray, prefix: str) -> None:
    with psycopg.connect(SRC) as c:
        register_vector(c)
        with c.cursor() as cur:
            cur.executemany(f"INSERT INTO {SCHEMA}.docs (title, embedding) VALUES (%s, %s)", [(f"{prefix} {i}", v) for i, v in enumerate(vectors)])


insert_docs(random_unit(N_DOCS), "doc")
q(f"SELECT 'orders' AS tbl, count(*) AS n FROM {SCHEMA}.orders UNION ALL SELECT 'docs', count(*) FROM {SCHEMA}.docs")

## 2. One statement

`CREATE JOB ... ON STREAMING FROM POSTGRES ... TO DATABASE` is the whole setup. The statement runs in
the context of a Doris database (the job belongs to it; without one Doris answers
`Unknown database ''`), and `replication_num = 1` is needed because this cluster has a single BE.
`offset = initial` copies the existing rows first; `latest` would only stream from now on.
`max_interval` is a job property, the idle wait between sync rounds; it matters in section 4.

The announcement says the source needs a publication named `dbz_publication` created `FOR ALL
TABLES`. In 4.1.4 that is not what happens: the job creates its own publication, listing only the
included tables, and its own pgoutput slot, both named after the job id.

In [ ]:
t0 = time.perf_counter()
create_job("orders_job", "orders")
s = wait_for(lambda: dcount("SELECT count(*) FROM orders") == N_ROWS)
print(f"job created and {N_ROWS:,} rows copied in {time.perf_counter() - t0:.1f}s")
display(job("orders_job"))
print("on the source:")
display(q("""
    SELECT p.pubname, p.puballtables, string_agg(t.schemaname || '.' || t.tablename, ', ') AS tables
    FROM pg_publication p LEFT JOIN pg_publication_tables t USING (pubname) GROUP BY 1, 2
"""))
q("SELECT slot_name, plugin, slot_type, active FROM pg_replication_slots")

## 3. What lands in Doris

The FE read the Postgres schema and created a `UNIQUE KEY` table (merge-on-write, so an update
replaces the row and a delete removes it). The type mapping, next to what the
[mapping page](https://doris.apache.org/docs/4.x/data-operate/import/import-way/streaming-job/data-type-mapping-postgresql/)
documents:

In [ ]:
pg_types = q("""
    SELECT column_name AS column,
           CASE WHEN data_type IN ('ARRAY', 'USER-DEFINED') THEN udt_name ELSE data_type END AS postgres
    FROM information_schema.columns
    WHERE table_schema = %s AND table_name = 'orders'
    ORDER BY ordinal_position
""", SCHEMA)
doris_types = dq("DESC orders")[["Field", "Type", "Key"]].rename(columns={"Field": "column", "Type": "doris", "Key": "key"})
documented = {"id": "BIGINT", "customer": "STRING", "status": "STRING", "amount": "DECIMAL(12, 2)", "payload": "JSON", "tags": "ARRAY<STRING>", "created_at": "DATETIME(6)"}
types = pg_types.merge(doris_types, on="column")
types["documented"] = types.column.map(documented)
types

Doris prints `STRING` as `text`. `jsonb` arrives as `text`, not the documented `JSON`. The values are intact, and Doris's JSON
functions take a string, so the analytics still work; what is lost is the binary JSON storage and the
type check on write. If that matters, create the Doris table yourself with a `JSON` column before
the job: an existing table is used as it is. `uuid` becomes `text`, as documented.

`timestamptz` becomes `datetime(6)`, which has no zone, "normalized to the server time zone". Both
containers run in UTC here, so the values line up; with a Doris cluster in another zone they shift.

The kind of query the copy is for, on the Doris side:

In [ ]:
display(dq("SELECT @@time_zone AS doris_time_zone"))
dq("""
    SELECT get_json_string(payload, '$.channel') AS channel, status,
           count(*) AS orders, sum(amount) AS revenue, avg(get_json_int(payload, '$.items')) AS avg_items
    FROM orders GROUP BY 1, 2 ORDER BY 1, 2
""")

## 4. Changes flow, and what sets the pace

Insert, update and delete on the source, then poll Doris until it agrees. Postgres to Postgres took
milliseconds in `logical_replication.ipynb`. Here the job works in rounds: the FE schedules a task,
the BE's CDC client reads what the slot has, writes it with a Stream Load, reports the offset, and
the FE schedules the next round after `max_interval` seconds when there was nothing to read. With
the default of 10 a change takes about 11 seconds to show up. This notebook runs with
`MAX_INTERVAL` (1 by default), and it still takes about 5: a round reads the stream for a few
seconds before it commits, so a change waits for the round it lands in to finish (section 7 lists
the rounds with their start and finish times). Seconds, not the milliseconds of a Postgres
subscriber; fine for dashboards, not for read-your-writes.

In [ ]:
with psycopg.connect(SRC) as c:
    c.execute(f"INSERT INTO {SCHEMA}.orders (status, amount, payload, tags) SELECT 'new', 1, '{{}}', '{{}}' FROM generate_series(1, 1000)")
s_insert = wait_for(lambda: dcount("SELECT count(*) FROM orders") == N_ROWS + 1000)

with psycopg.connect(SRC) as c:
    c.execute(f"UPDATE {SCHEMA}.orders SET status = 'refunded' WHERE mod(id, 10) = 0")
n_refunded = q(f"SELECT count(*) AS n FROM {SCHEMA}.orders WHERE status = 'refunded'").n[0]
s_update = wait_for(lambda: dcount("SELECT count(*) FROM orders WHERE status = 'refunded'") == n_refunded)

with psycopg.connect(SRC) as c:
    c.execute(f"DELETE FROM {SCHEMA}.orders WHERE id > %s", (N_ROWS + 900,))
s_delete = wait_for(lambda: dcount("SELECT count(*) FROM orders") == N_ROWS + 900)

pd.DataFrame({
    "change": ["1000 inserts", f"{n_refunded} updates", "100 deletes"],
    "visible in Doris after (s)": [round(x, 2) for x in (s_insert, s_update, s_delete)],
})

## 5. The `vector` column

The mapping page says other and user-defined types become `STRING`. The job disagrees, three ways.

**a. Auto table creation refuses the table.** The FE finds no mapping for `vector` and the
`CREATE JOB` fails, before anything is created.

In [ ]:
try:
    create_job("docs_job", "docs")
except pymysql.MySQLError as e:
    print(f"CREATE JOB refused: {e.args[1]}")

**b. A pre-created table is accepted, and the vectors arrive as NULL.** An existing target table is
used as it is, so creating `docs` in Doris first gets past the check. The load then reports no
errors and no filtered rows, and every `embedding` is NULL. The reason is only in the BE's
`cdc-client.log`: the CDC client is Debezium's Postgres connector (`io.debezium.*`), which has no
converter for pgvector's type and leaves the column out of every event.

The two `real[]` copies are ordinary Postgres types, so they come across as `ARRAY<FLOAT>`.

In [ ]:
dx(f"""
    CREATE TABLE docs (
        id BIGINT NOT NULL,
        title STRING,
        embedding STRING,
        embedding_gen ARRAY<FLOAT>,
        embedding_arr ARRAY<FLOAT>
    )
    UNIQUE KEY(id) DISTRIBUTED BY HASH(id) BUCKETS 1
    PROPERTIES ("replication_num" = "1")
""")
create_job("docs_job", "docs")
wait_for(lambda: dcount("SELECT count(*) FROM docs") == N_DOCS)
display(dq("""
    SELECT count(*) AS doc_rows, count(embedding) AS embedding, count(embedding_gen) AS embedding_gen,
           count(embedding_arr) AS embedding_arr, max(array_size(embedding_arr)) AS dim
    FROM docs
"""))
display(job("docs_job").loc[["Status", "LoadStatistic", "ErrorMsg"]])
log = subprocess.run(["docker", "exec", "doris-be", "grep", "-m1", "No converter found", "/opt/apache-doris/be/log/cdc-client.log"], capture_output=True, text=True).stdout
print(log.strip()[:400])

**c. The generated copy disappears once streaming starts.** The initial copy is a `SELECT`, which
reads generated columns. The stream is pgoutput, and Postgres 16 does not publish generated
columns (Postgres 18 can, with `publish_generated_columns`). When the first streamed change for
`docs` arrives without `embedding_gen`, Doris takes it as a schema change and drops the column from
its table (section 6 shows that schema changes do propagate). The trigger-maintained copy is a real
column in the WAL and survives.

In [ ]:
insert_docs(random_unit(10), "streamed")
wait_for(lambda: dcount("SELECT count(*) FROM docs") == N_DOCS + 10)
display(dq("DESC docs")[["Field", "Type"]])
dq("SELECT count(*) AS doc_rows, count(embedding_arr) AS embedding_arr FROM docs")

So the working recipe for embeddings is a trigger that keeps a `real[]` copy (pgvector casts
`vector` to `real[]` directly), plus a pre-created Doris table, because auto creation still refuses
the `vector` column itself. The copy doubles the embedding storage on the source, and the `vector`
column in Doris is dead weight that can be dropped.

With the copy in place, Doris can run the same nearest-neighbour query. `cosine_distance` on
`ARRAY<FLOAT>` is an exact scan here; Doris 4 also has ANN indexes on array columns, not used in this
notebook.

In [ ]:
qv = random_unit(1)[0]
pg_top = q(f"SELECT id, embedding <=> %s AS distance FROM {SCHEMA}.docs ORDER BY 2 LIMIT 5", qv)
literal = "[" + ", ".join(f"{x:.9g}" for x in qv) + "]"
doris_top = dq(f"SELECT id, cosine_distance(embedding_arr, {literal}) AS distance FROM docs ORDER BY distance LIMIT 5")
both = pg_top.merge(doris_top, on="id", how="outer", suffixes=("_pgvector", "_doris"))
print(f"same top 5: {sorted(pg_top.id) == sorted(doris_top.id)}, max |distance difference| = {np.abs(both.distance_pgvector - both.distance_doris).max():.2e}")
both

## 6. Schema changes propagate

The opposite of Postgres to Postgres, where a column added on the source stops the apply worker
until someone adds it on the replica. Here the job notices the new column in the stream and alters
the Doris table itself, and a dropped column is dropped. Section 5c is the same mechanism
misfiring.

In [ ]:
run(f"ALTER TABLE {SCHEMA}.orders ADD COLUMN coupon text", autocommit=True)
run(f"INSERT INTO {SCHEMA}.orders (status, amount, payload, tags, coupon) VALUES ('new', 10, '{{}}', '{{}}', 'WELCOME')")
s_add = wait_for(lambda: dcount("SELECT count(*) FROM orders WHERE coupon = 'WELCOME'") == 1)
print(f"ADD COLUMN: new column and row in Doris after {s_add:.1f}s")

run(f"ALTER TABLE {SCHEMA}.orders DROP COLUMN coupon", autocommit=True)
run(f"INSERT INTO {SCHEMA}.orders (status, amount, payload, tags) VALUES ('after drop', 10, '{{}}', '{{}}')")
s_drop = wait_for(lambda: dcount("SELECT count(*) FROM orders WHERE status = 'after drop'") == 1)
print(f"DROP COLUMN: row in Doris after {s_drop:.1f}s; Doris columns now: {', '.join(dq('DESC orders').Field)}")

## 7. Monitoring, and a paused job holds WAL

Doris side: `jobs()` for the job (status, `CurrentOffset` as an LSN, rows and bytes loaded) and
`tasks()` for the individual rounds. Postgres side: the job's slot in `pg_replication_slots`, which
is the number that decides whether the source's disk fills up. Every job has its own slot, so
Postgres decodes the same WAL once per job; group tables into one job where possible.

A paused job is exactly the forgotten subscriber from `logical_replication.ipynb`: its slot stops
advancing and pins WAL. Even a running job's slot lags by up to a round, because the offset is
confirmed to Postgres when a round finishes; the difference is that the paused one keeps growing. `max_slot_wal_keep_size` caps it on the source, at the price of an
invalidated slot and a job that has to start again.

In [ ]:
job_id = dq("SELECT Id FROM jobs('type'='insert') WHERE Name = 'orders_job'").Id[0]
tasks = dq(f"SELECT TaskId, Status, CreateTime, StartTime, FinishTime, LoadStatistic FROM tasks('type'='insert') WHERE JobId = '{job_id}'")
display(tasks.sort_values("CreateTime", ascending=False).head(4))

slots = """
    SELECT slot_name, active, pg_size_pretty(pg_wal_lsn_diff(pg_current_wal_lsn(), confirmed_flush_lsn)) AS retained
    FROM pg_replication_slots ORDER BY slot_name
"""
dx("PAUSE JOB WHERE jobName = 'orders_job'")
time.sleep(3)
run(f"INSERT INTO {SCHEMA}.orders (status, amount, payload, tags) SELECT 'burst', 1, '{{}}', '{{}}' FROM generate_series(1, %s)", N_BURST)
time.sleep(2)
print(f"orders_job paused, {N_BURST:,} rows written:")
display(q(slots))

dx("RESUME JOB WHERE jobName = 'orders_job'")
s = wait_for(lambda: dcount("SELECT count(*) FROM orders WHERE status = 'burst'") == N_BURST, timeout=300)
print(f"resumed: caught up in {s:.1f}s")
display(job("orders_job").loc[["Status", "CurrentOffset", "LoadStatistic"]])
q(slots)

## 8. What the announcement says, and what this run showed

| claim | on Doris 4.1.4 with Postgres 16 |
|---|---|
| one SQL statement | yes, run inside a Doris database, with `replication_num = 1` on a single BE, and the FE and BE memory capped on a laptop |
| no Debezium | no separate Debezium: the CDC client next to the BE is Debezium's Postgres connector |
| publication `dbz_publication` FOR ALL TABLES | the job creates its own, `doris_pub_<job id>`, for the included tables only, and a `doris_cdc_<job id>` slot |
| real time | seconds: about 11 with the default `max_interval` of 10, about 5 with 1, because a round itself lasts about 5 |
| type mapping as documented | `jsonb` lands as `text`, not `JSON` |
| user-defined types become `STRING` | `vector` is refused by auto creation, and silently NULL in a pre-created table |
| schema changes | added and dropped columns propagate, which pg to pg does not do |
| cleanup | `DROP JOB` removes its slot and its publication |

When it fits: keeping a set of Postgres tables queryable in Doris with no transformation on the way,
which is the ODS layer of a warehouse on Doris; models and aggregates are built on top of it inside
Doris. Embeddings need the `real[]` copy. When it does not: several consumers of the same changes,
enrichment or routing in flight, or a target other than Doris; that is still Debezium plus Kafka, or
Flink CDC. Built-in CDC is marked experimental in the Doris documentation.

## Cleanup

Dropping the jobs drops their slots and publications on the source; the check below should list
none. Then the Doris database and the Postgres schema. The containers stay up;
`docker compose -f docker/postgres-pgvector.yaml --profile doris stop doris-fe doris-be` stops Doris
alone, and `just db-down doris -v` removes everything in the file, pg16's data included.

In [ ]:
for name in ("orders_job", "docs_job"):
    dx(f"DROP JOB WHERE jobName = '{name}'")
time.sleep(5)
print("slots left on the source:", q("SELECT slot_name FROM pg_replication_slots").slot_name.tolist())
print("publications left:", q("SELECT pubname FROM pg_publication").pubname.tolist())
dx(f"DROP DATABASE IF EXISTS {DORIS_DB} FORCE", database=None)
run(f"DROP SCHEMA IF EXISTS {SCHEMA} CASCADE", autocommit=True)